# SciDCC Preprocessing

Ulaz: `data/processed/scidcc_clean.csv` iz EDA notebooka.

Izlaz: tokenizirani HuggingFace `DatasetDict` (train/val/test) spreman za `Trainer`, plus class weights za handling imbalancea.

Plan:

1. Ucitaj cisti CSV i label mapping
2. Stratified split 70/15/15, fiksni seed
3. Pripremi dvije verzije inputa za usporedbu: `title_summary` i `body_head`
4. Tokenizacija s CliSciBERT tokenizerom
5. Izracunaj class weights za CrossEntropyLoss
6. Spremi sve na disk

## 1. Setup i ucitavanje

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

SEED = 42
PROCESSED_DIR = Path('..') / 'data' / 'processed'

TOKENIZER_MODEL = 'P0L3/SciClimateBERT'
MAX_LENGTH = 512

In [3]:
df = pd.read_csv(PROCESSED_DIR / 'scidcc_clean.csv')

with open(PROCESSED_DIR / 'label_map.json', 'r', encoding='utf-8') as f:
    label_map = json.load(f)
label2id = label_map['label2id']
id2label = {int(k): v for k, v in label_map['id2label'].items()}

print(f'Shape: {df.shape}')
print(f'Klase:  {len(label2id)}')
df['label'] = df['Category'].map(label2id)
assert df['label'].notna().all(), 'nedostaju labeli'
df['label'] = df['label'].astype(int)
df[['Category', 'label']].head(3)

Shape: (11534, 14)
Klase:  20


,Category,label
0,Ozone Holes,16
1,Ozone Holes,16
2,Ozone Holes,16


## 2. Stratified split 70/15/15

Dvije klase imaju ispod 50 uzoraka (`Global Warming` 21, `Geology` 28). Stratified split ih moze ocuvati u svim setovima, ali uzorci u val/test ce biti mali (3 do 4 po setu). Prihvacam to jer je najiskrenije prema originalnom datasetu.

In [4]:
# prvo split na train vs temp (85% / 15% test)
train_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED
)
# iz train onda odvoji val: target je 15% od totala, sto je 15/85 od train
train_df, val_df = train_test_split(
    train_df, test_size=15/85, stratify=train_df['label'], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df):>5}  ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val:   {len(val_df):>5}  ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test:  {len(test_df):>5}  ({len(test_df)/len(df)*100:.1f}%)')

Train:  8073  (70.0%)
Val:    1730  (15.0%)
Test:   1731  (15.0%)


In [5]:
# provjera da su sve klase u svim splitovima
split_counts = pd.DataFrame({
    'train': train_df['Category'].value_counts(),
    'val':   val_df['Category'].value_counts(),
    'test':  test_df['Category'].value_counts(),
}).fillna(0).astype(int)
split_counts['total'] = split_counts.sum(axis=1)
split_counts = split_counts.sort_values('total', ascending=False)
split_counts

,train,val,test,total
Category,,,,
Earthquakes,688,148,148,984
Pollution,661,142,142,945
Genetically Modified,640,137,137,914
Agriculture & Food,590,127,127,844
Hurricanes Cyclones,590,126,126,842
Animals,530,114,114,758
Weather,503,108,108,719
Endangered Animals,491,105,105,701
Climate,489,105,105,699


In [6]:
# koliko minimalno uzoraka ima najmanja klasa u val i test?
print(f'Min u val:  {split_counts["val"].min()}  (klasa: {split_counts["val"].idxmin()})')
print(f'Min u test: {split_counts["test"].min()}  (klasa: {split_counts["test"].idxmin()})')

Min u val:  3  (klasa: Global Warming)
Min u test: 3  (klasa: Global Warming)


## 3. Priprema dvije input varijante

`title_summary`: konkatenacija Title + Summary, garantirano stane u 510 tokena.

`body_head`: prvih 512 tokena Body-a (head truncation). Koristi vise sadrzaja ali gubi sredinu i kraj dugackih clanaka.

Dvije varijante treniram odvojeno, pa u eval notebooku usporedim.

In [7]:
def build_title_summary(row):
    title = str(row['Title']).strip()
    summary = str(row['Summary']).strip()
    return f'{title}. {summary}' if summary else title

def build_body(row):
    return str(row['Body']).strip()

for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_df['text_title_summary'] = split_df.apply(build_title_summary, axis=1)
    split_df['text_body']          = split_df.apply(build_body, axis=1)

train_df[['Title', 'Summary', 'text_title_summary']].head(2)

,Title,Summary,text_title_summary
0,Ozone hole modest despite optimum conditions f...,The ozone hole that forms in the upper atmosph...,Ozone hole modest despite optimum conditions f...
1,"Swedish, Finnish and Russian wolves closely re...",The Scandinavian wolf originally came from Fin...,"Swedish, Finnish and Russian wolves closely re..."


## 4. Tokenizacija

Padding radim tek u DataCollator-u (dynamic padding) tokom treniranja, tu samo truncation. Spremam i `attention_mask` i `input_ids`.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
print(f'Tokenizer: {TOKENIZER_MODEL}')
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Max model input: {tokenizer.model_max_length}')

Tokenizer: P0L3/SciClimateBERT
Vocab size: 50265
Max model input: 512


In [ ]:
# helper za pretvaranje pandas splitova u HF Dataset s izabranim text poljem
def make_hf_dataset(df, text_col):
    out = df[[text_col, 'label']].rename(columns={text_col: 'text'})
    return Dataset.from_pandas(out, preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,  # dinamicki u collatoru
    )

def build_dataset_dict(text_col):
    raw = DatasetDict({
        'train': make_hf_dataset(train_df, text_col),
        'val':   make_hf_dataset(val_df,   text_col),
        'test':  make_hf_dataset(test_df,  text_col),
    })
    tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['text'])
    return tokenized

In [10]:
ds_title_summary = build_dataset_dict('text_title_summary')
ds_body          = build_dataset_dict('text_body')

print('title_summary:')
print(ds_title_summary)
print('\nbody:')
print(ds_body)

Map:   0%|          | 0/8073 [00:00<?, ? examples/s]

Map:   0%|          | 0/1730 [00:00<?, ? examples/s]

Map:   0%|          | 0/1731 [00:00<?, ? examples/s]

Map:   0%|          | 0/8073 [00:00<?, ? examples/s]

Map:   0%|          | 0/1730 [00:00<?, ? examples/s]

Map:   0%|          | 0/1731 [00:00<?, ? examples/s]

title_summary:
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 8073
    })
    val: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1730
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1731
    })
})

body:
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 8073
    })
    val: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1730
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1731
    })
})


In [11]:
# sanity check: koliko je prosjecna duljina input_ids nakon truncationa
for name, ds in [('title_summary', ds_title_summary), ('body', ds_body)]:
    lens = [len(x) for x in ds['train']['input_ids']]
    print(f'{name:15s}  mean={np.mean(lens):.1f}  p95={np.percentile(lens, 95):.0f}  max={np.max(lens)}')

title_summary    mean=75.2  p95=141  max=336
body             mean=477.6  p95=512  max=512


## 5. Class weights

Racunam samo nad train setom (kasnije iskoristim u CrossEntropyLoss). Koristim `balanced` shemu iz sklearn-a koja je inverzno proporcionalna frekvenciji klase.

In [12]:
classes = np.arange(len(label2id))
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['label'].values,
)

weights_df = pd.DataFrame({
    'class_id': classes,
    'category': [id2label[i] for i in classes],
    'train_count': [int((train_df['label'] == i).sum()) for i in classes],
    'weight': weights.round(4),
}).sort_values('train_count')
weights_df

,class_id,category,train_count,weight
12,12,Global Warming,15,26.9100
11,11,Geology,20,20.1825
19,19,Zoology,147,2.7459
8,8,Extinction,250,1.6146
14,14,Microbes,278,1.4520
10,10,Geography,285,1.4163
3,3,Biotechnology,322,1.2536
7,7,Environment,334,1.2085
15,15,New Species,369,1.0939
2,2,Biology,434,0.9301


## 6. Spremanje

Sve ide u `data/processed/`:

* `splits/train.csv, val.csv, test.csv` za evidenciju (koji id je u kojem splitu)
* `hf_title_summary/` i `hf_body/` kao HuggingFace on-disk datasetovi
* `class_weights.json` za loss funkciju

In [13]:
splits_dir = PROCESSED_DIR / 'splits'
splits_dir.mkdir(exist_ok=True, parents=True)

# samo bitne kolone, bez tokeniziranih polja
keep_cols = ['Date', 'Link', 'Title', 'Summary', 'Body', 'Category', 'Year', 'label']
train_df[keep_cols].to_csv(splits_dir / 'train.csv', index=False)
val_df[keep_cols].to_csv(splits_dir / 'val.csv', index=False)
test_df[keep_cols].to_csv(splits_dir / 'test.csv', index=False)
print(f'CSV splits saved to {splits_dir}')

ds_title_summary.save_to_disk(str(PROCESSED_DIR / 'hf_title_summary'))
ds_body.save_to_disk(str(PROCESSED_DIR / 'hf_body'))
print(f'HF datasets saved')

with open(PROCESSED_DIR / 'class_weights.json', 'w', encoding='utf-8') as f:
    json.dump({
        'classes':  [int(c) for c in classes],
        'weights':  [float(w) for w in weights],
        'scheme':   'balanced',
        'computed_on': 'train_split',
        'seed': SEED,
    }, f, indent=2)
print('Class weights saved')

CSV splits saved to ../data/processed/splits


Saving the dataset (0/1 shards):   0%|          | 0/8073 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1730 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1731 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8073 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1730 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1731 [00:00<?, ? examples/s]

HF datasets saved
Class weights saved


In [14]:
# sanity check: reload s diska i verifikacija
from datasets import load_from_disk
check = load_from_disk(str(PROCESSED_DIR / 'hf_title_summary'))
print(check)
print('\nPrimjer:')
ex = check['train'][0]
print(f'  label:       {ex["label"]} ({id2label[ex["label"]]})')
print(f'  input_ids:   {ex["input_ids"][:20]}...')
print(f'  length:      {len(ex["input_ids"])}')
print(f'  decoded:     {tokenizer.decode(ex["input_ids"])[:200]}...')

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 8073
    })
    val: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1730
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1731
    })
})

Primjer:
  label:       16 (Ozone Holes)
  input_ids:   [0, 673, 13930, 4683, 6473, 1135, 33771, 1437, 50272, 13, 34011, 39309, 4, 20, 34011, 4683, 14, 4620, 11, 5]...
  length:      43
  decoded:     <s>Ozone hole modest despite optimum conditions for ozone depletion. The ozone hole that forms in the upper atmosphere over Antarctica each September was slightly above average size in 2018, NOAA and ...
